In [189]:
import pandas as pd
import requests
import os
from calculadora_inflacao import CalculadoraFatorInflacao

# APOGESP
## Cálculo de impacto orçamentário das propostas reajuste salarial para os APPGGS

Esse notebook realiza a estimativa de impacto orçamentário das propostas de reajuste salarial feitas pela APOGESP para a campanha salarial de 2026 na Prefeitura de São Paulo.

São consideradas duas propostas principais:
1. Reajuste inflacionário, que se subdivide em:
    1. Reajuste da tabela original da carreira considerando a inflação acumulada desde as primeiras nomeações (jun/2026)
    2. Reajuste da tabela da carreira considerando a inflação acumulada na gestão Ricardo Nunes e a tabela no momento de início da gestão (jun/2021)
2. Equiparação com a tabela dos AMCI (ou seja, unificação da estrutura remuneratória do quadro QPPGG)


Em ambos os casos, a estimativa segue a seguinte metodologia:

1. Identificação dos APPGGs em exercício (incluso cedidos) segundo os últimos dados disponíveis no portal de dados abertos (dezembro de 2025),
2. Atualização dos dados do item 1 considerando o acréscimo dos 30 APPGGs recém-nomeados no nível 1, criando dados sintéticos (novas linhas na tabela)
3. Cálculo do salário base atual de cada APPGG considerando a tabela atual e o nível em que ele se encontra,
4. Cálculo dos encargos e benefícios que tem como base o salário atual como contribuição previdenciária obrigatória, auxílio transporte, auxílio alimentação etc.
5. Cálculo do salário proposto considerando o nível em que o APPGG se encontra atualmente e a tabela atualizada proposta
6. Cálculo dos encargos citados no item 6,
7. Somatória dos salários e encargos para cada APPGG considerando a tabela atual e a tabela proposta,
8. Cálculo da diferença entre a situação atual e a situação proposta conforme item 7,
9. Somatória da diferença por nível da carreira APPGG (p. ex., para o nível 1 a proposta implica em X mil reais mensais para a Prefeitura para o nível 2, y mil reais etc.)
10. Simulação de progressão da carreira considerando 18 meses para a "subida" de nível e projeção do impacto orçamentário anual até o final da gestão (2028)

O diagrama na imagem abaixo representa as etapas da metodologia.

![Grafo da metodologia](mermaid_graph.svg)

## Dados servidores

Nessa seção pegamos os dados dos servidores ativos mais atualizados (dezembro de 2025) do portal de dados abertos. Em seguida, identificamos os APPGGs e adicionamos dados sintéticos referentes aos APPGGs recém-nomeados.

In [190]:
URL_SERVIDORES = 'https://dados.prefeitura.sp.gov.br/dataset/bf5df0f4-4fb0-4a5e-b013-07d098cc7b1c/resource/9fd35d06-d861-4a19-9926-a0fb3de42b50/download/verificado_ativos_03-03-2026_fev-2026in.csv'
file_dados = 'servidores_ativos.csv'

In [191]:
def download_dados_servidores(fname:str, url:str=URL_SERVIDORES)->str:

    if not os.path.exists(fname):
        print('Downloading data')
        with requests.get(url) as r:
            r.raise_for_status()
            content = r.content
            with open(fname, 'wb') as f:
                f.write(content)
            
            assert os.path.exists(fname)
            return fname
    return fname

In [192]:
file_dados = download_dados_servidores(file_dados)

In [193]:
df = pd.read_csv(file_dados, encoding='latin1', sep=';')

In [194]:
df.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,1145541,16,CLEUZA BORGES PEREIRA SILVA,ASSESSOR IV,CDA-4,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1949,PARDA,NAO
1,1160206,5,ADILSON DA SILVA,COORDENADOR II,CDA-6,NaN,QC,CARGO EM COMISSAO,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTO AMARO,SUB-SA,COORDENADORIA DE PLANEJAMENTO E DESENVOLVIMENT...,SANTO AMARO,SANTO AMARO,NaN,MASCULINO,1949,BRANCA,NAO
2,1160478,2,JULIO DE CARVALHO,FISCAL DE POSTURAS MUNICIPAIS NIVEL IV,QFPM15,NaN,QFPM,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SUBPREFEITURA SANTANA/TUCURUVI,SUB-ST,SUPERVISÃO TECNICA DE FISCALIZACAO,TUCURUVI,SANTANA/TUCURUVI,NaN,MASCULINO,1951,BRANCA,NAO
3,1161181,9,NEUSA PEDRAO NASSIR,ASSESSOR V,CDA-5,NaN,QC,CARGO EM COMISSAO,NAO SE APLICA,NaN,...,SECRETARIA DO GOVERNO MUNICIPAL,SGM,ASSESSORIA JURIDICA,SE,SE,NaN,FEMININO,1947,BRANCA,NAO
4,1168185,4,MARIA HELENA DI VERNIERI CUPPARI,ASSISTENTE TECNICO EDUCACIONAL,QPE17A,NaN,QPE L. 14660/07 RGPS,CARGO EM COMISSAO,LICENCIATURA PLENA COMPLETA,NaN,...,SECRETARIA MUNICIPAL DE EDUCACAO,SME,DIRETORIA REGIONAL DE EDUCACAO PIRITUBA/JARAGUA,LAPA,LAPA,NaN,FEMININO,1942,BRANCA,NAO


In [195]:
cargos = df['REF_CARGO_BAS'].unique()

In [196]:
for cargo in cargos:
    if "appgg" in str(cargo).lower():
        print(cargo)

APPGG2
APPGG1
APPGG5
APPGG6
APPGG3
APPGG4


In [197]:
#filtrando para apenas servidores efetivos (ou seja, que possuem cargo base)
df_efetivos = df[df['REF_CARGO_BAS'].notna()]

In [198]:
appggs = df_efetivos[df_efetivos['REF_CARGO_BAS'].str.contains('APPGG')].reset_index(drop=True)

In [199]:
appggs.head()

,REGISTRO,VINCULO,NOME,CARGO_BASICO,REF_CARGO_BAS,SEGMENTO,GRUPO,SUBGRUPO,ESCOL_CARGO_BASICO,CARGO_COMISSAO,...,SECRET_SUBPREF,SIGLA,SETOR,DISTRITO,SUBPREFEITURA,ORGAO_EXT,SEXO,ANO_NASCIMENTO,RACA_COR,PCD
0,6394604,3,CLAUDIO AGUIAR ALMEIDA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SMC,SECRETARIA MUNICIPAL DE CULTURA E ECONOMIA CRI...,SE,SE,NaN,MASCULINO,1963,PRETA,NAO
1,7575491,2,MAURICIO DA SILVA CORREIA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG1,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE GESTAO,SEGES,SECRETARIA MUNICIPAL DE GESTAO,SE,SE,PREFEITURA DO MUNICIPIO DE ITAJAI,MASCULINO,1986,BRANCA,NAO
2,7718543,6,MARCIA MIYUKI ISHIKAWA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE HABITACAO,SEHAB,DEPARTAMENTO DE PLANEJAMENTO HABITACIONAL,SE,SE,NaN,FEMININO,1978,BRANCA,NAO
3,7794720,2,TIAGO ROSA MACHADO,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG2,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,ASSESSOR III,...,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,SEME,SECRETARIA MUNICIPAL DE ESPORTES E LAZER,MOEMA,VILA MARIANA,NaN,MASCULINO,1983,BRANCA,NAO
4,7840501,2,THAIS ROBERTO DA SILVA,ANALISTA POLITICAS PUBLICAS GESTAO GOVERNAMENT...,APPGG5,NaN,QPGG,SUPERIOR,SUPERIOR COMPLETO,NaN,...,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SMDHC,SECRETARIA MUNICIPAL DE DIREITOS HUMANOS E CID...,SE,SE,NaN,FEMININO,1977,PRETA,NAO


In [200]:
appggs = appggs[['REGISTRO', 'NOME', 'REF_CARGO_BAS', 'SIGLA', 'DATA_INICIO_EXERC']]

renomear_cols = {
    'REGISTRO' : 'rf',
    'NOME' : 'nome',
    'REF_CARGO_BAS' : 'cargo_base',
    'SIGLA' : 'secretaria_dez_2025',
    'DATA_INICIO_EXERC' : 'dt_inicio_exercicio'
}

appggs.rename(renomear_cols, axis=1, inplace=True)


In [201]:
appggs['nivel_carreira'] = appggs['cargo_base'].str.extract(r'(\d+)$').astype(int)

In [202]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,01/10/2021,2
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,03/11/2021,1
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,08/12/2021,2
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,05/01/2022,2
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,29/11/2017,5


In [203]:
appggs.shape

(185, 6)

In [204]:
QTD_RECEM_NOMEADOS=13
recem_nomeados = [{
            'rf' : str(i+1).zfill(7),
            'nome' : f'recem_nomeado_{i+1}',
            'cargo_base' : 'APPGG1',
            'secretaria_dez_2025' : 'NA',
            'nivel_carreira' : 1,
            'dt_inicio_exercicio' : '01/03/2026'
            }
            for i in range(30)
            ]

recem_nomeados = pd.DataFrame(recem_nomeados)

In [205]:
recem_nomeados.head()

,rf,nome,cargo_base,secretaria_dez_2025,nivel_carreira,dt_inicio_exercicio
0,0000001,recem_nomeado_1,APPGG1,NA,1,01/03/2026
1,0000002,recem_nomeado_2,APPGG1,NA,1,01/03/2026
2,0000003,recem_nomeado_3,APPGG1,NA,1,01/03/2026
3,0000004,recem_nomeado_4,APPGG1,NA,1,01/03/2026
4,0000005,recem_nomeado_5,APPGG1,NA,1,01/03/2026


In [206]:
recem_nomeados.shape

(30, 6)

In [207]:
appggs = pd.concat([appggs, recem_nomeados])

In [208]:
appggs.shape

(215, 6)

In [209]:
#vou colocar como datetime porque precisamos calcular com base na data
appggs['dt_inicio_exercicio'] = pd.to_datetime(appggs['dt_inicio_exercicio'], format ="%d/%m/%Y")


#inclusive ja vou definir se a pessoa contribui para o regime proprio (IPREM) ou nao
appggs['contribui_rpps'] = appggs['dt_inicio_exercicio'].dt.year<2018

In [210]:
appggs.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True


# Situaçao atual

Nessa seção calculamos a situação atual de dispêndio com a carreira de APPGGs por parte da Prefeitura, considerando apenas os vencimentos e os encargos que estão relacionados aos vencimentos, como contribuição previdenciaria.

In [211]:
tabela_atual = {
    1 : 13208.14,
    2 : 14528.96,
    3 : 14892.18,
    4 : 15264.48,
    5 : 15646.09,
    6 : 16037.24,
    7 : 17961.73,
    8 : 18410.77,
    9 : 18871.04,
    10 : 19342.80,
    11 : 19826.39,
    12 : 21869.74,
    13 : 22416.47,
    14 : 22976.89,
    15 : 23551.31
}

In [212]:
appggs_atual = appggs.copy(deep=True)

In [213]:
appggs_atual['vencimento'] = appggs_atual['nivel_carreira'].map(tabela_atual)

In [214]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09


### Terço adicional de férias e décimo terceiro

Nesse caso vamos dividir ambos por 12 são gastos anuais e nossa base é mensal.

In [215]:
def calcular_decimo_terceiro(row):

    return round(row['vencimento']/12, 2)

def calcular_terco_ferias(row):

    return round((row['vencimento']/3)/12, 2)

In [216]:
appggs_atual['decimo_terceiro'] = appggs_atual.apply(calcular_decimo_terceiro, axis=1)

appggs_atual['terco_ferias'] = appggs_atual.apply(calcular_terco_ferias, axis=1)

In [217]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96,1210.75,403.58
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14,1100.68,366.89
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96,1210.75,403.58
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96,1210.75,403.58
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61


### Vale alimentação

O vale alimentação só é devido para quem ganha menos de 10 salários minimos. Como com o reajuste essa proporcao pode diminuir, ele deve entrar na conta apesar de ser um valor fixo.

In [218]:
def calcular_va(row):
    SALARIO_MINIMO = 1621
    VALOR_VALE_ALIMENTACAO = 325.25

    if row['vencimento'] <= SALARIO_MINIMO*10:
        return VALOR_VALE_ALIMENTACAO
    return 0

In [219]:
appggs_atual['vale_alimentacao'] = appggs_atual.apply(calcular_va, axis=1)

In [220]:
appggs_atual.sample(3)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao
3,0000004,recem_nomeado_4,APPGG1,NA,2026-03-01,1,False,13208.14,1100.68,366.89,325.25
26,8358877,JULIANA HELENA BONAT,APPGG6,SEGES,2016-07-19,6,True,16037.24,1336.44,445.48,325.25
108,8905720,ALEXANDRE YUUZO TANABE,APPGG2,SMDHC,2021-10-07,2,False,14528.96,1210.75,403.58,325.25


### Contribuicao previdenciaria

Para calcular a contribuicao previdenciaria precisamos saber se a pessoa entrou antes de 2018 (ou seja, se é do IPREM) ou depois (se é do SAMPAPREV)

In [221]:
def valor_iprem(row):

    CONTRIBUICAO_IPREM = 0.28

    if not row['contribui_rpps']:
        return 0
    
    # o iprem nao considera o terço adicional de férias
    valor_base = row['vencimento'] + row['decimo_terceiro']
    
    return round(valor_base * CONTRIBUICAO_IPREM, 2)

In [222]:
appggs_atual['contribuicao_iprem'] = appggs_atual.apply(valor_iprem, axis=1)

In [223]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem
37,8359024,BRENDA MACHADO FONSECA,APPGG6,SEGES,2016-06-15,6,True,16037.24,1336.44,445.48,325.25,4864.63
183,9537929,IGOR ROCHA DA SILVA,APPGG1,SEGES,2026-02-06,1,False,13208.14,1100.68,366.89,325.25,0.00
152,9411810,HELOISA PINHEIRO ROSA DE CASTRO,APPGG1,SEGES,2024-07-17,1,False,13208.14,1100.68,366.89,325.25,0.00
32,8358958,GRAZIELA MARTINS PEDRO DIAS,APPGG6,SEGES,2016-07-08,6,True,16037.24,1336.44,445.48,325.25,4864.63


In [224]:
def contribuicao_inss(row):

    ALIQUOTA_INSS = 0.21
    TETO_INSS = 8475.55

    if row['contribui_rpps']:
        return 0
    
    # o inss vai sobre todos os vencimentos, incluso terco adicional
    valor_base = row['vencimento'] + row['terco_ferias'] + row['decimo_terceiro']

    #só para validar o teto - no caso de APPGG da na mesma porque tá todo mundo acima, mas pra deixar a funcao correta
    if valor_base > TETO_INSS:
        valor_base = TETO_INSS
        
    return round(valor_base*ALIQUOTA_INSS, 2)
    


In [225]:
appggs_atual['contribuicao_inss'] = appggs_atual.apply(contribuicao_inss, axis=1)

In [226]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss
64,8383898,DAVI GIUGNO,APPGG6,SEGES,2017-01-18,6,True,16037.24,1336.44,445.48,325.25,4864.63,0.00
15,0000016,recem_nomeado_16,APPGG1,NA,2026-03-01,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87
83,8549966,MARCUS VINICIUS FERREIRA DA CUNHA CASASCO,APPGG1,SEGES,2026-02-06,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87
110,8905746,GABRIELA SANTOS NEVES,APPGG2,SEGES,2021-11-03,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87


In [227]:
def previdencia_complementar(row):

    TETO_INSS = 8475.55
    ALIQUOTA_COMPLEMENTAR = 0.075
    if row['contribui_rpps']:
        return 0
    
    valor_base = row['vencimento'] + row['decimo_terceiro']
    #só contribui o que é acima do teto
    valor_base = valor_base - TETO_INSS

    return round(valor_base * ALIQUOTA_COMPLEMENTAR, 2)



In [228]:
appggs_atual['previdencia_complementar'] = appggs_atual.apply(previdencia_complementar, axis=1)

In [229]:
appggs_atual.sample(4)

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar
90,8893756,GABRIELA EVANGELISTA DE SOUZA,APPGG1,SEHAB,2026-02-13,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.5
49,8359237,MARILIA ROMAO CAPINZAIKI,APPGG6,SEPLAN,2016-07-01,6,True,16037.24,1336.44,445.48,325.25,4864.63,0.00,0.0
5,0000006,recem_nomeado_6,APPGG1,NA,2026-03-01,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.5
51,8359407,ELIZETE REGINA NICOLINI,APPGG6,SGM,2016-07-11,6,True,16037.24,1336.44,445.48,325.25,4864.63,0.00,0.0


## Valor total

Agora calculamos o valor total e salvamos os dados

In [230]:
def valor_total_prefeitura(row):

    vencimentos = row['vencimento'] + row['decimo_terceiro'] + row['terco_ferias']
    auxilios = row['vale_alimentacao']
    previdencia = row['contribuicao_iprem'] + row['contribuicao_inss'] + row['previdencia_complementar']

    return round(vencimentos+auxilios+previdencia, 2)

In [231]:
appggs_atual['valor_total_prefeitura'] = appggs_atual.apply(valor_total_prefeitura, axis=1)

In [232]:
appggs_atual.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.50,17218.33
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61,325.25,4745.98,0.00,0.00,22455.77


In [233]:
appggs_atual.to_csv('situacao_atual.csv', sep=';')

# Proposta equiparação AMCI

Agora calculamos quais seriam as contribuicoes da Prefeitura com base na equiparação com a tabela dos AMCI

In [234]:
tabela_amci = {
    1 : 16840.38,
    2 : 17682.41,
    3 : 18036.04,
    4 : 18396.78,
    5 : 18764.71,
    6 : 19139.99,
    7 : 20097.00,
    8 : 20498.94,
    9 : 20908.92,
    10 : 21327.10,
    11 : 21753.64,
    12 : 22841.33,
    13 : 23183.95,
    14 : 23531.72,
    15 : 23884.67
}

In [235]:
proposta_amci = appggs.copy(deep=True)

In [236]:
proposta_amci.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True


Agora vamos calcular rapidamente o que fizemos na seção anterior, qual seja:
1. o novo vencimento,
2. o décimo terceiro,
3. o terço adicional de férias,
4. o auxilio alimentacao,
5. as contribuicoes previdenciarias

In [237]:
#vencimento
proposta_amci['vencimento'] = proposta_amci['nivel_carreira'].map(tabela_amci)
#ferias e decimo terceiro
proposta_amci['decimo_terceiro'] = proposta_amci.apply(calcular_decimo_terceiro, axis=1)
proposta_amci['terco_ferias'] = proposta_amci.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_amci['vale_alimentacao'] = proposta_amci.apply(calcular_va, axis=1)
#previdencia
proposta_amci['contribuicao_iprem'] = proposta_amci.apply(valor_iprem, axis=1)
proposta_amci['contribuicao_inss'] = proposta_amci.apply(contribuicao_inss, axis=1)
proposta_amci['previdencia_complementar'] = proposta_amci.apply(previdencia_complementar, axis=1)

#VALOR TOTAL PROPOSTA
proposta_amci['valor_total_prefeitura'] = proposta_amci.apply(valor_total_prefeitura, axis=1)

In [238]:
proposta_amci.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,17682.41,1473.53,491.18,0,0.00,1779.87,801.03,22228.02
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,16840.38,1403.37,467.79,0,0.00,1779.87,732.62,21224.03
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,17682.41,1473.53,491.18,0,0.00,1779.87,801.03,22228.02
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,17682.41,1473.53,491.18,0,0.00,1779.87,801.03,22228.02
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,18764.71,1563.73,521.24,0,5691.96,0.00,0.00,26541.64


In [239]:
proposta_amci.to_csv('proposta_amci.csv', sep=';')

# Proposta reajuste inflação
### Reajuste tabela original

In [240]:
tabela_original = {
    1 : 9000.00,
    2 : 10080.00,
    3 : 10684.80,
    4 : 11325.89,
    5 : 12005.44,
    6 : 12725.77,
    7 : 13998.34,
    8 : 14698.26,
    9 : 15433.17,
    10 : 16204.83,
    11 : 17015.08,
    12 : 18716.58,
    13 : 19558.83,
    14 : 20438.98,
    15 : 21358.73
}

In [241]:
calculadora_inflacao = CalculadoraFatorInflacao('ipc-fipe')

In [242]:
# nesse momento o ultimo valor do IPCA disponível é para janeiro de 2026
fator_ipc = calculadora_inflacao('01/06/2016', '01/01/2026')

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F06%2F2016&dataFinal=01%2F01%2F2026


In [243]:
fator_ipc

1.5697766159124333

In [244]:
fator_ipca = CalculadoraFatorInflacao('ipca')('01/06/2016', '01/01/2026')
fator_ipca

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json&dataInicial=01%2F06%2F2016&dataFinal=01%2F01%2F2026


1.5887292779061

In [245]:
tabela_atualizada = {nivel : round(valor*fator_ipc, 2) for nivel, valor in tabela_original.items()}
tabela_atualizada

{1: 14127.99,
 2: 15823.35,
 3: 16772.75,
 4: 17779.12,
 5: 18845.86,
 6: 19976.62,
 7: 21974.27,
 8: 23072.98,
 9: 24226.63,
 10: 25437.96,
 11: 26709.87,
 12: 29380.85,
 13: 30702.99,
 14: 32084.63,
 15: 33528.43}

In [246]:
proposta_ipc = appggs.copy(deep=True)

In [247]:
#vencimento
proposta_ipc['vencimento'] = proposta_ipc['nivel_carreira'].map(tabela_atualizada)
#ferias e decimo terceiro
proposta_ipc['decimo_terceiro'] = proposta_ipc.apply(calcular_decimo_terceiro, axis=1)
proposta_ipc['terco_ferias'] = proposta_ipc.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_ipc['vale_alimentacao'] = proposta_ipc.apply(calcular_va, axis=1)
#previdencia
proposta_ipc['contribuicao_iprem'] = proposta_ipc.apply(valor_iprem, axis=1)
proposta_ipc['contribuicao_inss'] = proposta_ipc.apply(contribuicao_inss, axis=1)
proposta_ipc['previdencia_complementar'] = proposta_ipc.apply(previdencia_complementar, axis=1)
#VALOR TOTAL PROPOSTA
proposta_ipc['valor_total_prefeitura'] = proposta_ipc.apply(valor_total_prefeitura, axis=1)

In [248]:
proposta_ipc.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,15823.35,1318.61,439.54,325.25,0.00,1779.87,649.98,20336.60
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,14127.99,1177.33,392.44,325.25,0.00,1779.87,512.23,18315.11
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,15823.35,1318.61,439.54,325.25,0.00,1779.87,649.98,20336.60
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,15823.35,1318.61,439.54,325.25,0.00,1779.87,649.98,20336.60
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,18845.86,1570.49,523.50,0.00,5716.58,0.00,0.00,26656.43


In [249]:
proposta_ipc.to_csv('proposta_ipc.csv', sep=';')

### Reajuste tabela Gestão Nunes

In [250]:
# agora consideramos junho de 2021 como o ano base
# vamos usar a tabela original como base porque só teve reajuste de 0.001% nesse período então nem compensa atualizar

In [251]:
fator_ipc_nunes = calculadora_inflacao('01/06/2021', '01/01/2026')

Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F06%2F2021&dataFinal=01%2F01%2F2026


In [252]:
fator_ipc_nunes = calculadora_inflacao('01/06/2021', '01/01/2026')


Dados obtidos com sucesso na URL: https://api.bcb.gov.br/dados/serie/bcdata.sgs.193/dados?formato=json&dataInicial=01%2F06%2F2021&dataFinal=01%2F01%2F2026


In [253]:
fator_ipc_nunes

1.2888145550973353

In [254]:
tabela_atualizada_nunes = {nivel : round(valor*fator_ipc_nunes, 2) for nivel, valor in tabela_original.items()}
tabela_atualizada_nunes

{1: 11599.33,
 2: 12991.25,
 3: 13770.73,
 4: 14596.97,
 5: 15472.79,
 6: 16401.16,
 7: 18041.26,
 8: 18943.33,
 9: 19890.49,
 10: 20885.02,
 11: 21929.28,
 12: 24122.2,
 13: 25207.7,
 14: 26342.05,
 15: 27527.44}

Só a comparação com a tabela atual permite identificar que alguns níveis já tiveram reajuste até acima da inflação da gestão. Então no caso vamos manter esses niveis como estão.

In [255]:
tabela_atual

{1: 13208.14,
 2: 14528.96,
 3: 14892.18,
 4: 15264.48,
 5: 15646.09,
 6: 16037.24,
 7: 17961.73,
 8: 18410.77,
 9: 18871.04,
 10: 19342.8,
 11: 19826.39,
 12: 21869.74,
 13: 22416.47,
 14: 22976.89,
 15: 23551.31}

In [256]:
tabela_atualizada_nunes = {nivel : valor if valor>tabela_atual[nivel] else tabela_atual[nivel]
                           for nivel, valor in tabela_atualizada_nunes.items()}

In [257]:
tabela_atualizada_nunes

{1: 13208.14,
 2: 14528.96,
 3: 14892.18,
 4: 15264.48,
 5: 15646.09,
 6: 16401.16,
 7: 18041.26,
 8: 18943.33,
 9: 19890.49,
 10: 20885.02,
 11: 21929.28,
 12: 24122.2,
 13: 25207.7,
 14: 26342.05,
 15: 27527.44}

In [258]:
proposta_ipc_nunes = appggs.copy(deep=True)

In [259]:
#vencimento
proposta_ipc_nunes['vencimento'] = proposta_ipc_nunes['nivel_carreira'].map(tabela_atualizada_nunes)
#ferias e decimo terceiro
proposta_ipc_nunes['decimo_terceiro'] = proposta_ipc_nunes.apply(calcular_decimo_terceiro, axis=1)
proposta_ipc_nunes['terco_ferias'] = proposta_ipc_nunes.apply(calcular_terco_ferias, axis=1)
#vale alimentacao
proposta_ipc_nunes['vale_alimentacao'] = proposta_ipc_nunes.apply(calcular_va, axis=1)
#previdencia
proposta_ipc_nunes['contribuicao_iprem'] = proposta_ipc_nunes.apply(valor_iprem, axis=1)
proposta_ipc_nunes['contribuicao_inss'] = proposta_ipc_nunes.apply(contribuicao_inss, axis=1)
proposta_ipc_nunes['previdencia_complementar'] = proposta_ipc_nunes.apply(previdencia_complementar, axis=1)
#VALOR TOTAL PROPOSTA
proposta_ipc_nunes['valor_total_prefeitura'] = proposta_ipc_nunes.apply(valor_total_prefeitura, axis=1)

In [260]:
proposta_ipc_nunes.head()

,rf,nome,cargo_base,secretaria_dez_2025,dt_inicio_exercicio,nivel_carreira,contribui_rpps,vencimento,decimo_terceiro,terco_ferias,vale_alimentacao,contribuicao_iprem,contribuicao_inss,previdencia_complementar,valor_total_prefeitura
0,6394604,CLAUDIO AGUIAR ALMEIDA,APPGG2,SMC,2021-10-01,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
1,7575491,MAURICIO DA SILVA CORREIA,APPGG1,SEGES,2021-11-03,1,False,13208.14,1100.68,366.89,325.25,0.00,1779.87,437.50,17218.33
2,7718543,MARCIA MIYUKI ISHIKAWA,APPGG2,SEHAB,2021-12-08,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
3,7794720,TIAGO ROSA MACHADO,APPGG2,SEME,2022-01-05,2,False,14528.96,1210.75,403.58,325.25,0.00,1779.87,544.81,18793.22
4,7840501,THAIS ROBERTO DA SILVA,APPGG5,SMDHC,2017-11-29,5,True,15646.09,1303.84,434.61,325.25,4745.98,0.00,0.00,22455.77


In [261]:
proposta_ipc_nunes.to_csv('proposta_ipc_nunes.csv', sep=';')

In [262]:
def tabela_to_pandas(tabela:dict, tabela_name:str):

    dados = []
    for nivel, val in tabela.items():
        dados.append([tabela_name, nivel, val])

    return pd.DataFrame(dados, columns = ['nome_tabela', 'nivel', 'vencimento'])

In [263]:
tabelas = {
    'atual' : tabela_atual,
    'original' : tabela_original,
    'amci' : tabela_amci,
    'original_atualizada_ipc' : tabela_atualizada,
    'original_atualizada_ipc_nunes' : tabela_atualizada_nunes

}

In [264]:
dfs_tabelas = [tabela_to_pandas(tabela, tabela_name) for tabela_name, tabela in tabelas.items()]
df_tabelas = pd.concat(dfs_tabelas)

df_tabelas

,nome_tabela,nivel,vencimento
0,atual,1,13208.14
1,atual,2,14528.96
2,atual,3,14892.18
3,atual,4,15264.48
4,atual,5,15646.09
...,...,...,...
10,original_atualizada_ipc_nunes,11,21929.28
11,original_atualizada_ipc_nunes,12,24122.20
12,original_atualizada_ipc_nunes,13,25207.70
13,original_atualizada_ipc_nunes,14,26342.05


In [265]:
df_tabelas.to_csv('tabelas_vencimentos_utilizadas.csv', sep=';')

# Agregações

Agora fazemos as agregações para analisar o impacto orçamentário

In [266]:
def total_mensal_por_nivel(servidores, nome_proposta:str):

    df = servidores.groupby('nivel_carreira')['valor_total_prefeitura'].sum().reset_index()
    df['id_proposta'] = nome_proposta
    return df

In [267]:
total_atual = total_mensal_por_nivel(appggs_atual, 'situacao_atual')
total_atual

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1706403.77,situacao_atual
1,2,1127593.20,situacao_atual
2,3,40615.73,situacao_atual
3,4,21916.01,situacao_atual
4,5,356961.79,situacao_atual
5,6,851334.48,situacao_atual


In [268]:
total_amci = total_mensal_por_nivel(proposta_amci, 'unificacao_amci')
total_amci

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,2103774.73,unificacao_amci
1,2,1333681.20,unificacao_amci
2,3,48160.64,unificacao_amci
3,4,26021.22,unificacao_amci
4,5,421643.12,unificacao_amci
5,6,1001681.02,unificacao_amci


In [269]:
total_ipc = total_mensal_por_nivel(proposta_ipc, 'reajuste_ipc')
total_ipc

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1815189.28,reajuste_ipc
1,2,1220196.00,reajuste_ipc
2,3,44867.50,reajuste_ipc
3,4,25147.57,reajuste_ipc
4,5,423461.73,reajuste_ipc
5,6,1045465.71,reajuste_ipc


In [270]:
total_ipc_nunes = total_mensal_por_nivel(proposta_ipc_nunes, 'reajuste_ipc_gestao_nunes')
total_ipc_nunes

,nivel_carreira,valor_total_prefeitura,id_proposta
0,1,1706403.77,reajuste_ipc_gestao_nunes
1,2,1127593.20,reajuste_ipc_gestao_nunes
2,3,40615.73,reajuste_ipc_gestao_nunes
3,4,21916.01,reajuste_ipc_gestao_nunes
4,5,356961.79,reajuste_ipc_gestao_nunes
5,6,858345.61,reajuste_ipc_gestao_nunes


In [271]:
def impacto_mensal(situacao_atual, proposta):

    join = pd.merge(situacao_atual, proposta, on='nivel_carreira', suffixes=('_atual', '_proposta'))
    join['impacto_mensal'] = join['valor_total_prefeitura_proposta'] - join['valor_total_prefeitura_atual']

    return join



In [272]:
impacto_mensal_amci = impacto_mensal(total_atual, total_amci)
impacto_mensal_ipca = impacto_mensal(total_atual, total_ipc)
impacto_mensal_ipca_nunes = impacto_mensal(total_atual, total_ipc_nunes)

In [273]:
impacto_mensal_amci.to_csv('impacto_mensal_amci.csv', sep=';')
impacto_mensal_amci

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1706403.77,situacao_atual,2103774.73,unificacao_amci,397370.96
1,2,1127593.20,situacao_atual,1333681.20,unificacao_amci,206088.00
2,3,40615.73,situacao_atual,48160.64,unificacao_amci,7544.91
3,4,21916.01,situacao_atual,26021.22,unificacao_amci,4105.21
4,5,356961.79,situacao_atual,421643.12,unificacao_amci,64681.33
5,6,851334.48,situacao_atual,1001681.02,unificacao_amci,150346.54


In [274]:
impacto_mensal_ipca.to_csv('impacto_mensal_ipc.csv', sep=';')
impacto_mensal_ipca

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1706403.77,situacao_atual,1815189.28,reajuste_ipc,108785.51
1,2,1127593.20,situacao_atual,1220196.00,reajuste_ipc,92602.80
2,3,40615.73,situacao_atual,44867.50,reajuste_ipc,4251.77
3,4,21916.01,situacao_atual,25147.57,reajuste_ipc,3231.56
4,5,356961.79,situacao_atual,423461.73,reajuste_ipc,66499.94
5,6,851334.48,situacao_atual,1045465.71,reajuste_ipc,194131.23


In [275]:
impacto_mensal_ipca_nunes.to_csv('impacto_mensal_ipc_nunes.csv', sep=';')
impacto_mensal_ipca_nunes

,nivel_carreira,valor_total_prefeitura_atual,id_proposta_atual,valor_total_prefeitura_proposta,id_proposta_proposta,impacto_mensal
0,1,1706403.77,situacao_atual,1706403.77,reajuste_ipc_gestao_nunes,0.00
1,2,1127593.20,situacao_atual,1127593.20,reajuste_ipc_gestao_nunes,0.00
2,3,40615.73,situacao_atual,40615.73,reajuste_ipc_gestao_nunes,0.00
3,4,21916.01,situacao_atual,21916.01,reajuste_ipc_gestao_nunes,0.00
4,5,356961.79,situacao_atual,356961.79,reajuste_ipc_gestao_nunes,0.00
5,6,851334.48,situacao_atual,858345.61,reajuste_ipc_gestao_nunes,7011.13


In [276]:
impactos_anuais = {
'reajuste_ipca_gestao_nunes' : round(impacto_mensal_ipca_nunes['impacto_mensal'].sum()*12, 2),
'reajuste_ipca' : round(impacto_mensal_ipca['impacto_mensal'].sum()*12, 2),
'reajuste_amci' : round(impacto_mensal_amci['impacto_mensal'].sum()*12, 2),
}

In [277]:
tabela_to_pandas(impactos_anuais, 'impactos_anuais').drop('nome_tabela', axis=1).to_csv('impactos_anuais.csv', sep=';')